In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/annotated/checkpoints/post_cell_6.pickle

trying: ['file_name_2017', 'file_name_2018']
me:  8
me:  8
trying: ['file_name_2017', 'file_name_2018']
me:  8
me:  8
trying: ['base_dir_2021']
me:  8
trying: ['file_name_2021']
me:  8
trying: ['base_dir_2019']
me:  8
trying: ['np']
me:  0
trying: ['base_dir_2018']
me:  8
trying: ['grab_subset_of_data']
me:  4
trying: ['responses_df_2017']


me:  8
trying: ['base_dir_2020']
me:  8
trying: ['pd']
me:  0
trying: ['base_dir_2017']
me:  8
trying: ['responses_df_2020']


me:  8
trying: ['bar_chart_multiple_years']
me:  2
trying: ['responses_df_2021']


me:  8
trying: ['create_choropleth_map']
me:  2
trying: ['directory']
me:  8
trying: ['combine_subset_of_data_from_multiple_years']
me:  6
trying: ['bar_chart_multiple_choice_multiple_selection']
me:  2
trying: ['base_dir_2022']
me:  8
trying: ['combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions']
me:  6
trying: ['bar_chart_multiple_choice']
me:  2
trying: ['file_name_2022']
me:  8
trying: ['warnings']
me:  0
trying: ['convert_df_of_counts_to_percentages']
me:  4
trying: ['px']
me:  0
trying: ['create_dataframe_of_counts']
me:  4
trying: ['file_name_2019']
me:  8
trying: ['sns']
me:  0
trying: ['count_then_return_percent']
me:  4
trying: ['add_year_column_to_dataframes']
me:  4
trying: ['load_survey_data']
me:  6
trying: ['glob']
me:  0
trying: ['file_name_2020']
me:  8
trying: ['responses_df_2022']


me:  10
trying: ['responses_df_2018']


me:  10
trying: ['orig_output']
me:  13
trying: ['replace_hyphen_with_en_dash']
me:  10
trying: ['responses_df_2019']


me:  10
trying: ['go']
me:  0
Declaring variable np
Declaring variable pd
Declaring variable warnings
Declaring variable px
Declaring variable sns
Declaring variable glob
Declaring variable go
Declaring variable bar_chart_multiple_years
Declaring variable create_choropleth_map
Declaring variable bar_chart_multiple_choice_multiple_selection
Declaring variable bar_chart_multiple_choice
Declaring variable grab_subset_of_data
Declaring variable convert_df_of_counts_to_percentages
Declaring variable create_dataframe_of_counts
Declaring variable count_then_return_percent
Declaring variable add_year_column_to_dataframes
Declaring variable combine_subset_of_data_from_multiple_years
Declaring variable combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions
Declaring variable load_survey_data
Declaring variable file_name_2017
Declaring variable file_name_2018
Declaring variable file_name_2017
Declaring variable file_name_2018
Declaring variable base_dir_2021
De

In [4]:
%%RecordEvent
%%time
### cell 7 ###

# cell 7 optimized
# Slice once with .iloc to avoid multiple __getitem__ calls
filtered = responses_df_2022.iloc[1:]
col = 'In which country do you currently reside?'

# Compute percentages directly on GPU with a single value_counts(normalize=True) + mul
percentages_per_country_df = (
    filtered[col]
    .value_counts(normalize=True)       # GPU: get proportions
    .mul(100)                           # GPU: convert to percentages
    .reset_index()                      # GPU: turn Series into DataFrame
    .rename({'index': 'country', col: '% of respondents'}, axis=1)  # GPU: rename columns
)

# Preserve the title and legend label
title_for_chart = 'Percentage of total responses for most common countries in 2022'
label_for_legend = '% of respondents'

# Write out the CSV (GPU)
percentages_per_country_df.to_csv(
    '/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/'
    + title_for_chart + '.csv',
    index=True
)

print('Note that countries with less than 50 responses were replaced with the country name "other" (which does not show up on this map)')

Note that countries with less than 50 responses were replaced with the country name "other" (which does not show up on this map)
CPU times: user 141 ms, sys: 16.2 ms, total: 157 ms
Wall time: 163 ms


In [5]:
%Checkpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high/checkpoints/post_cell_7_try_6.pickle

migration speed (bps): 787276807.8932586
---------------------------
variables to migrate:
create_choropleth_map 144
col 90
convert_df_of_counts_to_percentages 144
create_dataframe_of_counts 144
filtered 48
count_then_return_percent 144
add_year_column_to_dataframes 144
glob 72
grab_subset_of_data 144
combine_subset_of_data_from_multiple_years 144
combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions 144
load_survey_data 144
label_for_legend 65
base_dir_2021 155
percentages_per_country_df 48
file_name_2021 81
base_dir_2019 155
title_for_chart 112
file_name_2017 76
base_dir_2018 155
responses_df_2017 48
base_dir_2020 155
orig_output 48
base_dir_2017 155
responses_df_2020 48
responses_df_2021 48
directory 170
base_dir_2022 155
file_name_2022 81
file_name_2018 76
file_name_2019 78
file_name_2020 81
responses_df_2019 48
go 72
responses_df_2018 48
warnings 72
replace_hyphen_with_en_dash 144
np 72
responses_df_2022 48
px 72
sns 72
pd 72
bar_chart_multiple

In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables []
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables []
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 3 =======
Input variables []
Active variables []
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 4 =======
Input variables []
Active variables ['responses_df_2022', 'responses_df_2018', 'responses_df_2019']
Intermediate variables ['file_name_2021', 'base_dir_2017', 'base_dir_2018', 'responses_df_2017', 'file_name_2019', 'base_dir_2019', 'base_dir_2022', 'responses_df_2021', 'responses_df_2020', 'file_name_2017', 'directory', 'file_name_2020', 'file_name_2018', 'file_name_2022', 'base_dir_2021', 'base_dir_2020']
Future variables []
Modified dataframes
======= Cell 5 

In [7]:

with open("/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_7_try_6.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[7], f)


In [8]:
opt_output = Out.get(4)

In [9]:
assert False, 'title_for_chart is incorrectly modified in the optimized code.'

AssertionError: title_for_chart is incorrectly modified in the optimized code.